In [ ]:
#| default_exp review

## Reviewing notebooks

Diff and style feedback that focuses on code cells.

Review should focus on the behavior encoded in code cells, not on notebook metadata churn. This notebook keeps that review loop small: run fast.ai style hints when desired, and print nbdev-style code diffs when comparing notebooks.

In [ ]:
#| export
import json
import subprocess
from pathlib import Path

from chkstyle.core import main as _chkstyle_main
from fastcore.script import Param, call_parse
from nbdev.diff import nbs_pair, source_diff

from nbskill.foundation import cli_error, cli_return, none_if_string, tracked_call

### Style feedback as a tool

`style_check` wraps the fast.ai style checker so it can be called from the CLI or MCP without each caller rebuilding command arguments or handling strict mode.

In [ ]:
#| export
def _style_check_argv(path=".", skip_folder_re=None, skip_path=None):
    path = "." if path is None else str(path)
    argv = ["style_check", path]
    if skip_folder_re: argv += ["--skip-folder-re", str(skip_folder_re)]
    if skip_path: argv += ["--skip-path", str(skip_path)]
    return argv

In [ ]:
#| export
def run_style_check(path=".", skip_folder_re=None, skip_path=None, strict=False):
    status = _chkstyle_main(_style_check_argv(path, skip_folder_re, skip_path))
    if strict and status: raise SystemExit(status)
    return status

In [ ]:
#| export
@call_parse
@tracked_call
def style_check(
    path: Param("File or folder to check", str, opt=False, nargs="?") = ".",  # File or folder to check
    skip_folder_re: str | None = None,  # Regex for folders to skip
    skip_path: str | None = None,  # Folder name/path to skip
    strict: bool = False,  # Exit non-zero when style hints are found
):
    "Print fast.ai style hints using fastaistyle/chkstyle."
    status = run_style_check(path, skip_folder_re, skip_path, strict)
    return cli_return(status)

In [ ]:
#| export
def code_source(cell): return cell.source if cell.cell_type == "code" else None

### Code-cell diffs

Notebook diffs are noisy when metadata and outputs are included. `diff_nb` asks nbdev for code-cell source on each side of a comparison and prints only the added, changed, or deleted code blocks the caller requested.

In [ ]:
#| export
def _git_ref_path_error(path, ref):
    if ref is None: return None
    path = Path(path)
    root_cmd = subprocess.run(
        ["git", "-C", str(path.parent), "rev-parse", "--show-toplevel"],
        capture_output=True, text=True,
    )
    if root_cmd.returncode != 0:
        return f"No git repository found for {str(path)!r}."
    root = Path(root_cmd.stdout.strip())
    try:
        rel = path.resolve().relative_to(root.resolve()).as_posix()
    except ValueError:
        return f"{str(path)!r} is outside git repository {str(root)!r}."
    spec = f"{ref}:{rel}"
    exists_cmd = subprocess.run(
        ["git", "-C", str(root), "cat-file", "-e", spec],
        capture_output=True, text=True,
    )
    if exists_cmd.returncode == 0: return None
    return (
        f"Could not find notebook {rel!r} at git ref {ref!r}. "
        "The notebook may be new relative to that ref, or the repository may not have a HEAD commit yet. "
        "Commit the notebook first, choose an existing ref/path, or pass --ref_a None to compare against the working tree."
    )


def _git_root_rel(path):
    path = Path(path)
    root_cmd = subprocess.run(
        ["git", "-C", str(path.parent), "rev-parse", "--show-toplevel"],
        capture_output=True, text=True,
    )
    if root_cmd.returncode != 0: return None, None
    root = Path(root_cmd.stdout.strip())
    try: return root, path.resolve().relative_to(root.resolve()).as_posix()
    except ValueError: return None, None


def _notebook_json_at_ref(path, ref):
    path = Path(path)
    if ref is None:
        return json.loads(path.read_text(encoding="utf-8"))
    root, rel = _git_root_rel(path)
    if root is None: return None
    show = subprocess.run(["git", "-C", str(root), "show", f"{ref}:{rel}"], capture_output=True, text=True)
    if show.returncode != 0: return None
    return json.loads(show.stdout)


def _nbskill_metadata_by_cell(nb_json):
    cells = (nb_json or {}).get("cells", [])
    return {
        cell.get("id", str(idx)): (cell.get("metadata", {}) or {}).get("nbskill")
        for idx, cell in enumerate(cells)
    }


def _nbskill_metadata_change_count(path, ref_a, ref_b):
    try:
        old = _nbskill_metadata_by_cell(_notebook_json_at_ref(path, ref_a))
        new = _nbskill_metadata_by_cell(_notebook_json_at_ref(path, ref_b))
    except (OSError, json.JSONDecodeError, TypeError):
        return 0
    keys = set(old) | set(new)
    return sum(1 for key in keys if old.get(key) != new.get(key) and (old.get(key) is not None or new.get(key) is not None))


def _metadata_summary(count):
    if not count: return ""
    noun = "cell" if count == 1 else "cells"
    return f"Ignored nbskill metadata changes in {count} {noun}."


@call_parse
@tracked_call
def diff_nb(
    path: str,  # Notebook path
    ref_a: str|None = "HEAD",  # First git ref; use None for working tree
    ref_b: str|None = None,  # Second git ref; defaults to working tree
    adds: bool = True,  # Include code cells added in ref_b
    changes: bool = True,  # Include changed code cells
    dels: bool = False,  # Include deleted code cells
):
    "Print nbdev-style diffs for code cells only; summarize nbskill metadata-only changes."
    ref_a, ref_b = none_if_string(ref_a), none_if_string(ref_b)
    if msg := (_git_ref_path_error(path, ref_a) or _git_ref_path_error(path, ref_b)):
        cli_error(msg)
    try:
        old, new = nbs_pair(path, ref_a=ref_a, ref_b=ref_b, f=code_source)
    except Exception as exc:
        detail = str(exc)
        hint = (
            f"Could not diff {path!r} against {ref_a!r}. "
            "The notebook may be new relative to that git ref, or the repository may not have a HEAD commit yet. "
            "Commit the notebook first, or pass --ref_a None to compare against the working tree."
        )
        if detail: hint += f"\nUnderlying error: {detail}"
        cli_error(hint)
    old = {cid: src for cid, src in old.items() if src is not None}
    new = {cid: src for cid, src in new.items() if src is not None}
    blocks = []
    if adds:    blocks += [(cid, source_diff("", new[cid])) for cid in new if cid not in old]
    if changes: blocks += [(cid, source_diff(old[cid], new[cid])) for cid in new if cid in old and new[cid] != old[cid]]
    if dels:    blocks += [(cid, source_diff(old[cid], "")) for cid in old if cid not in new]
    text = "\n\n".join(f"--- code cell {cid} ---\n{diff}" for cid, diff in blocks if diff.strip())
    metadata_summary = _metadata_summary(_nbskill_metadata_change_count(path, ref_a, ref_b))
    if text and metadata_summary: report = f"{text}\n\n{metadata_summary}"
    elif text: report = text
    elif metadata_summary: report = f"No code cell changes\n{metadata_summary}"
    else: report = "No code cell changes"
    print(report)
    return cli_return(report)

In [ ]:
import tempfile as _tempfile
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
from pathlib import Path as _Path

from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import read_nb as _read_nb
from fastcore.nbio import write_nb as _write_nb
from nbskill.review import diff_nb

with _tempfile.TemporaryDirectory() as td:
    path = _Path(td) / "demo.ipynb"
    _write_nb(new_nb([mk_cell("x = 1", cell_type="code")]), path)
    try:
        diff_nb(str(path))
    except (SystemExit, ValueError) as exc:
        if isinstance(exc, SystemExit): assert exc.code == 1
        else: assert "No git repository" in str(exc)

with _tempfile.TemporaryDirectory() as td:
    root = _Path(td)
    path = root / "demo.ipynb"
    _write_nb(new_nb([mk_cell("x = 1", cell_type="code")]), path)
    import subprocess as _subprocess
    _subprocess.run(["git", "init"], cwd=root, check=True, capture_output=True)
    _subprocess.run(["git", "add", "demo.ipynb"], cwd=root, check=True, capture_output=True)
    _subprocess.run([
        "git", "-c", "user.name=Nbskill", "-c", "user.email=nbskill@example.com",
        "commit", "-m", "base",
    ], cwd=root, check=True, capture_output=True)
    nb = _read_nb(path)
    nb.cells[0].metadata["nbskill"] = {"cell_type": "code", "semantic_types": [], "source_hash": "demo"}
    _write_nb(nb, path)
    out = _StringIO()
    with _redirect_stdout(out):
        diff_nb(str(path))
    text = out.getvalue()
    assert "No code cell changes" in text
    assert "Ignored nbskill metadata changes in 1 cell" in text

In [ ]:
from fastcore.nbio import mk_cell
from nbskill.review import code_source

cell = mk_cell("x = 1", cell_type="code")
assert code_source(cell) == "x = 1"